# 15 — Play, inspect decisions, and watch real recordings

**Status: UNEXECUTED FIRST DRAFT.** Purpose: inspect a genuinely trained AHR v2 XL checkpoint, fresh public environment decisions and saved observed replays. Inputs: a trusted checkpoint produced by notebooks 10/11, public local ARC assets, and configured game-family splits. Outputs: an ipywidgets/Matplotlib dashboard, `reports/demo/<episode_id>/` public decision journals, and a final replay JSONL. Expected runtime: checkpoint loading and one inference must be profiled; fresh episodes are bounded to 64 actions / ten minutes. Run All loads an existing trained checkpoint and displays controls. It does **not** start fresh actions automatically.

Checkpoint directory: `checkpoints/<run_id>/` from training; this notebook never creates replacement random weights. Exact resume: rerun to load the same trusted checkpoint. Saved replays can resume playback at any recorded step. Interrupted fresh episodes cannot exactly restore hidden ARC state and are marked interrupted; start a new episode or replay the observed journal. The model is frozen for inference. It does not learn from watch controls or change benchmark weights.

Fresh play and recorded playback are labelled separately. Board panels contain actual public observations. Worker/Scout candidates and Lorentz event coordinates are model hypotheses; energies and predicted success are not verified game outcomes. The real environment supplies completion. Public demos are not private competition evidence.

## Load explicit interfaces; never execute training dependencies

In [ ]:
from pathlib import Path
import json
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'notebooks' / '00_setup_and_hardware.ipynb').exists()), None)
if ROOT is None:
    raise RuntimeError('Open a kernel in /workspace/arc-agi-3-ahrv2 or its notebooks directory.')
_nb = json.loads((ROOT / 'notebooks' / '00_setup_and_hardware.ipynb').read_text())
for _cell in _nb['cells']:
    if _cell['cell_type'] == 'code' and 'ahr-export' in _cell.get('metadata', {}).get('tags', []):
        exec(compile(''.join(_cell['source']), '00_setup_and_hardware.ipynb', 'exec'), globals())
load_exports([14], globals())

## Resolve a trained checkpoint
An explicit `cfg.demo.checkpoint_path` is preferred. Otherwise the known joint/supervised run manifests are checked for a hash-valid **best** trained checkpoint with step > 0. No directory-wide random file selection or random-model fallback is used. Loading pickle checkpoints is restricted to your own trusted project artifacts.

In [ ]:
import asyncio
import html

ARC_PALETTE = ('#FFFFFF','#CCCCCC','#999999','#666666','#333333','#000000',
               '#E53AA3','#FF7BCC','#F93C31','#1E93FF','#88D8F1','#FFDC00',
               '#FF851B','#921231','#4FCC30','#A356D6')

def resolve_demo_checkpoint(cfg, root=ROOT):
    root=Path(root)
    explicit=cfg.get('demo',{}).get('checkpoint_path')
    if explicit:
        path=Path(explicit).expanduser()
        path=path if path.is_absolute() else root/path
        if not path.is_file():
            raise FileNotFoundError(f'Configured trained checkpoint does not exist: {path}')
        return path
    for run_id in ('xl-joint-seed42-v1','xl-supervised-seed42-v1'):
        manifest_path=root/'checkpoints'/run_id/'manifest.json'
        if not manifest_path.exists():
            continue
        manifest=json.loads(manifest_path.read_text())
        record=manifest.get('best')
        if record is None or int(record.get('step',0))<=0:
            continue
        name=record.get('file','')
        if Path(name).name!=name:
            raise ValueError('Unsafe checkpoint filename in manifest.')
        path=manifest_path.parent/name
        if path.is_file() and sha256_file(path)==record.get('sha256'):
            return path
        raise ValueError(f'Best checkpoint hash mismatch: {path}')
    raise FileNotFoundError('No trained best checkpoint is available. Complete genuine demonstration/training and its evaluation, or set cfg.demo.checkpoint_path to your trusted trained artifact. No random fallback was allocated.')

def _json_portable(value):
    if isinstance(value,torch.Tensor):
        return value.detach().cpu().tolist()
    if isinstance(value,np.ndarray):
        return value.tolist()
    if isinstance(value,dict):
        return {str(key):_json_portable(item) for key,item in value.items()}
    if isinstance(value,(list,tuple)):
        return [_json_portable(item) for item in value]
    if isinstance(value,np.generic):
        return value.item()
    if value is None or isinstance(value,(str,int,float,bool)):
        return value
    raise TypeError(f'Unsupported demo journal value: {type(value).__name__}')

def _scorecard_view(scorecard,operation_mode):
    if scorecard is None:
        return None
    score=getattr(scorecard,'score',None)
    return {'score':float(score) if score is not None else None,
            'scope':'official-online-scorecard' if operation_mode in {'ONLINE','COMPETITION'} else 'public-offline-toolkit-score',
            'private_competition_result':False}

## Render actual observations, role energies, and event coordinates
The 16-color palette follows the [official rendering utility](https://github.com/arcprize/ARC-AGI/blob/main/arc_agi/rendering.py). Coordinates are shown as a time versus x projection of the model's 1+3-dimensional workspace. Edges indicate stored parent links, not proof of a game's causal rules. Queen allocation is displayed alongside the separately learned Worker/Scout energy scales; the chart never invents absent traces.

In [ ]:
def render_ahr_decision(row, title='Recorded public transition'):
    import matplotlib.pyplot as plt
    from matplotlib.colors import ListedColormap
    from IPython.display import display
    before=validate_grid(row.get('observed_grid',row.get('observation')))
    after=validate_grid(row.get('next_observed_grid',row.get('next_observation')))
    action=ActionSpec.from_dict(row['action'])
    trace=row.get('planning_trace',row.get('trace',{})) or {}
    figure,axes=plt.subplots(2,3,figsize=(15,9),constrained_layout=True)
    cmap=ListedColormap(ARC_PALETTE)
    for axis,grid,label in ((axes[0,0],before,'Observed BEFORE action'),(axes[0,1],after,'Observed AFTER action')):
        axis.imshow(grid,cmap=cmap,vmin=0,vmax=15,interpolation='nearest',origin='upper')
        axis.set_title(label);axis.set_xlabel('x');axis.set_ylabel('y')
        if action.action==6:
            axis.scatter([action.x],[action.y],s=120,marker='x',color='black',linewidths=2)
    axes[0,0].set_title(f'Observed BEFORE ACTION{action.action}' if action.action else 'Observed BEFORE RESET')
    candidates=trace.get('candidates',[])
    energy_axis=axes[0,2]
    for role,color,marker in (('worker','#2364AA','o'),('scout','#E07A1F','^')):
        selected=[(index,c) for index,c in enumerate(candidates) if c.get('role')==role and 'energy' in c]
        if selected:
            energy_axis.scatter([index for index,_ in selected],[float(c['energy']) for _,c in selected],label=role,color=color,marker=marker,s=24)
    energy_axis.set_title('Recorded conditional energies (imagined candidates)')
    energy_axis.set_xlabel('Candidate index');energy_axis.set_ylabel('Energy; scales are learned per role')
    if energy_axis.collections:
        energy_axis.legend()
    else:
        energy_axis.text(.5,.5,'No AHR candidate energy trace recorded',ha='center',va='center',transform=energy_axis.transAxes)
    queen=trace.get('queen',{})
    allocation_axis=axes[1,0]
    if queen:
        values=[float(queen.get('worker_weight',0)),float(queen.get('scout_weight',0))]
        allocation_axis.bar(['Worker','Scout'],values,color=['#2364AA','#E07A1F'])
        allocation_axis.set_ylim(0,max(1,max(values)*1.1))
        allocation_axis.set_ylabel('Queen allocation weight')
        allocation_axis.set_title(f"Queen | halt p={float(queen.get('halt_probability',0)):.3f}")
    else:
        allocation_axis.text(.5,.5,'Queen trace unavailable',ha='center',va='center',transform=allocation_axis.transAxes)
        allocation_axis.set_title('No inferred allocation shown')
    events=row.get('workspace_events',[])
    event_axis=axes[1,1]
    indexed={event['id']:event for event in events if len(event.get('coordinates',[]))==4}
    for event in indexed.values():
        point=event['coordinates']
        parent=indexed.get(event.get('parent_id'))
        if parent is not None:
            parent_point=parent['coordinates']
            event_axis.plot([parent_point[1],point[1]],[parent_point[0],point[0]],color='#BBBBBB',linewidth=.6)
    for kind,color,marker in (('observed','#2A7F40','s'),('imagined','#8558A5','o')):
        points=[event['coordinates'] for event in indexed.values() if event.get('kind')==kind]
        if points:
            event_axis.scatter([point[1] for point in points],[point[0] for point in points],color=color,marker=marker,label=kind,s=18)
    event_axis.set_title('Latent event DAG projection — model representation')
    event_axis.set_xlabel('Workspace x');event_axis.set_ylabel('Workspace t')
    if event_axis.collections:
        event_axis.legend()
    else:
        event_axis.text(.5,.5,'Coordinates were not stored in this replay',ha='center',va='center',transform=event_axis.transAxes)
    detail=axes[1,2];detail.axis('off')
    budget=trace.get('budget',{})
    text=[f"Mode: {row.get('mode','recorded replay')}",f"Game: {row.get('game_id','unknown')}",
          f"Action: {action.to_dict()}",f"Actual state: {row.get('state','not stored')}",
          f"Actual levels: {row.get('levels_completed','not stored')}",
          f"Decision seconds: {row.get('decision_seconds',trace.get('elapsed_seconds','not stored'))}",
          f"Model calls: {budget.get('used','not stored')} / {budget.get('maximum','not stored')}",
          f"By role: {budget.get('by_role',{})}",f"Imagined candidates: {len(candidates)}",
          'Predicted success is not observed completion.']
    detail.text(0,1,'\n'.join(text),va='top',wrap=True,fontsize=10)
    figure.suptitle(title,fontsize=14)
    display(figure);plt.close(figure)
    return figure

def load_watch_replay(path):
    path=Path(path)
    if path.is_dir():
        manifest_path=path/'manifest.json'
        if not manifest_path.exists():
            raise FileNotFoundError('Demo directory has no manifest.')
        manifest=json.loads(manifest_path.read_text())
        rows=[]
        for record in manifest.get('steps',[]):
            item=path/record['file']
            if Path(record['file']).name!=record['file'] or sha256_file(item)!=record['sha256']:
                raise ValueError('Demo replay step filename/hash mismatch.')
            rows.append(json.loads(item.read_text()))
    else:
        rows=[]
        with path.open(encoding='utf-8') as stream:
            for line_number,line in enumerate(stream,1):
                if line.strip():
                    try:
                        rows.append(json.loads(line))
                    except Exception as exc:
                        raise ValueError(f'Replay JSON error at {path}:{line_number}') from exc
    if not rows:
        raise ValueError('Replay contains no observed decisions.')
    for row in rows:
        validate_grid(row.get('observed_grid',row.get('observation')))
        validate_grid(row.get('next_observed_grid',row.get('next_observation')))
        ActionSpec.from_dict(row['action'])
        if row.get('imagined_is_evidence',False):
            raise ValueError('Replay incorrectly claims imagined state as evidence.')
        row['mode']='recorded replay — no environment actions'
    return rows

## Fresh frozen-checkpoint play and atomic observed journals
`FreshDemoPlayer` and `play_fresh` are real environment functions, not a replay presented as live. They pass the actual public action mask to `AHRModel.act`, keep explicit episode memory, and write the observed transition and returned role trace after every real action. A lost response is not retried. The journal records checkpoint identity and labels interruptions; it cannot exact-resume the external episode. Default UI operation is OFFLINE public development.

In [ ]:
class FreshDemoPlayer:
    def __init__(self,model,cfg,root=ROOT,max_actions=64,max_wall_seconds=600,operation_mode='OFFLINE'):
        self.model,self.cfg,self.root=model,copy.deepcopy(cfg),Path(root)
        self.max_actions,self.max_wall_seconds=int(max_actions),float(max_wall_seconds)
        if not 1<=self.max_actions<=256 or not 0<self.max_wall_seconds<=600:
            raise ValueError('Fresh demo must be bounded to <=256 actions and <=600 seconds.')
        identity=getattr(model,'checkpoint_identity',None)
        if not identity or int(identity.get('step',0))<=0:
            raise RuntimeError('Fresh demo requires a real trained checkpoint identity; no untrained fallback.')
        self.identity=copy.deepcopy(identity)
        self.operation_mode=str(operation_mode).upper()
        self.registry=SplitRegistry(cfg)
        self.session,self.observation,self.episode_state=None,None,None
        self.manifest,self.directory,self.started=None,None,None
        self.finished=False
        self.model.eval()

    def start(self,game_id):
        if self.session is not None:
            raise RuntimeError('Finish the active fresh episode first.')
        if self.registry.split_for(game_id) is None:
            raise ValueError('Fresh demo game must be in the configured public train/dev/heldout registry.')
        require_recovery_gate(root=self.root)
        episode_cfg=copy.deepcopy(self.cfg)
        episode_cfg['environment']['max_actions']=self.max_actions
        self.session=ARCSession(episode_cfg,operation_mode=self.operation_mode)
        try:
            self.observation=self.session.reset(game_id)
            self.episode_state=None
            self.started=time.monotonic();self.finished=False
            self.directory=self.root/'reports'/'demo'/self.observation['episode_id']
            self.manifest={'schema_version':1,'mode':'fresh-trained-checkpoint','checkpoint':self.identity,
                'game_id':self.observation['game_id'],'split':self.registry.split_for(game_id),
                'steps':[],'status':'running','operation_mode':self.operation_mode,
                'exact_environment_resume_supported':False,'imagined_is_evidence':False,
                'max_actions':self.max_actions,'max_wall_seconds':self.max_wall_seconds}
            atomic_json(self.directory/'manifest.json',self.manifest)
            return copy.deepcopy(self.observation)
        except BaseException:
            self.session.close();self.session=None
            raise

    def step(self):
        if self.session is None or self.finished:
            raise RuntimeError('Start an active fresh episode first.')
        if terminal_observation(self.observation):
            self.finish('terminal');return None
        if len(self.manifest['steps'])>=self.max_actions or time.monotonic()-self.started>=self.max_wall_seconds:
            self.finish('budget_truncated');return None
        if not self.observation['available_actions']:
            self.finish('no-public-actions');return None
        before=copy.deepcopy(self.observation)
        started=time.monotonic()
        try:
            action,state,trace=self.model.act(before,before['available_actions'],episode_state=self.episode_state,
                                             budget=int(self.model.cfg['reasoning']['max_model_calls']))
            decision_seconds=time.monotonic()-started
            after=self.session.step(action)
            self.episode_state=state
            trace=copy.deepcopy(trace)
            trace['planner_status']=trace.get('status')
            trace['status']='trained-checkpoint-inference'
            events=[{key:event.get(key) for key in ('id','kind','branch','parent_id','coordinates')}
                    for event in state.get('workspace',{}).get('events',[])]
            row={'schema_version':1,'episode_id':before['episode_id'],'step':len(self.manifest['steps']),
                 'game_id':before['game_id'],'split':self.manifest['split'],'mode':'fresh trained checkpoint',
                 'observed_grid':before['grid'].tolist(),'next_observed_grid':after['grid'].tolist(),
                 'action':action.to_dict(),'available_actions':before['available_actions'],
                 'next_available_actions':after['available_actions'],'levels_completed':after['levels_completed'],
                 'state':after['state'],'decision_seconds':decision_seconds,'planning_trace':_json_portable(trace),
                 'workspace_events':_json_portable(events),'checkpoint':self.identity,'imagined_is_evidence':False,
                 'provenance':{'kind':'observed','interface':'official-arc-agi-public-frame','guid':after.get('guid')}}
            filename=f"step-{row['step']:06d}.json"
            path=self.directory/filename
            atomic_json(path,row)
            self.manifest['steps'].append({'file':filename,'sha256':sha256_file(path)})
            self.observation=after
            atomic_json(self.directory/'manifest.json',self.manifest)
            if terminal_observation(after):
                self.finish('terminal')
            return row
        except BaseException as exc:
            self.finish('interrupted-'+type(exc).__name__)
            raise

    def finish(self,reason='user_stopped'):
        if self.manifest is None or self.finished:
            return self.manifest
        self.finished=True
        scorecard=None
        try:
            if self.session is not None:
                scorecard=self.session.close()
        finally:
            self.session=None
            self.manifest.update(status=str(reason),elapsed_seconds=time.monotonic()-self.started,
                actual_state=self.observation['state'],levels_completed=self.observation['levels_completed'],
                scorecard=_scorecard_view(scorecard,self.operation_mode))
            atomic_json(self.directory/'manifest.json',self.manifest)
            rows=load_watch_replay(self.directory) if self.manifest['steps'] else []
            _atomic_jsonl(self.directory/'replay.jsonl',rows)
        return self.manifest

def play_fresh(model,cfg,game_id,on_decision=None,root=ROOT,max_actions=64,max_wall_seconds=600,operation_mode='OFFLINE'):
    player=FreshDemoPlayer(model,cfg,root,max_actions,max_wall_seconds,operation_mode)
    player.start(game_id)
    try:
        while not player.finished:
            row=player.step()
            if row is not None and on_decision is not None:
                on_decision(row)
        return player.manifest
    finally:
        if not player.finished:
            player.finish('interrupted')

## Notebook dashboard
Replay controls move through recorded decisions without contacting ARC or calling the model. Fresh controls start a new real public game, let you step once or watch bounded autoplay, and stop between decisions. The interface labels both modes and never presents a saved success as a fresh outcome. Model calls are synchronous; Stop takes effect after the current bounded decision returns.

In [ ]:
class AHRWatchDashboard:
    def __init__(self,model,cfg,root=ROOT):
        import ipywidgets as widgets
        self.model,self.cfg,self.root=model,copy.deepcopy(cfg),Path(root)
        self.player,self.task,self.rows=None,None,[]
        self.stop_requested=False
        registry=SplitRegistry(cfg)
        games=[game for split in ('heldout','dev','train') for game in registry.splits[split]]
        if not games:
            raise ValueError('Configure public game-family splits before opening fresh-watch controls.')
        self.game=widgets.Dropdown(options=games,description='Fresh game:')
        self.start_button=widgets.Button(description='Start fresh OFFLINE',button_style='primary')
        self.step_button=widgets.Button(description='One AHR decision')
        self.play_button=widgets.Button(description='Watch fresh autoplay')
        self.stop_button=widgets.Button(description='Stop fresh',button_style='warning')
        self.output=widgets.Output()
        self.status=widgets.HTML(value='<b>Fresh controls idle. No actions have been taken.</b>')
        self.replay_path=widgets.Text(description='Replay path:',layout=widgets.Layout(width='95%'))
        self.load_button=widgets.Button(description='Load recorded replay')
        self.slider=widgets.IntSlider(min=0,max=0,value=0,description='Replay step:')
        self.replay_play=widgets.Play(min=0,max=0,value=0,interval=400)
        widgets.jslink((self.replay_play,'value'),(self.slider,'value'))
        self.slider.observe(self._replay_step,names='value')
        self.start_button.on_click(self._start)
        self.step_button.on_click(self._step)
        self.play_button.on_click(self._play)
        self.stop_button.on_click(self._stop)
        self.load_button.on_click(self._load)
        self.widget=widgets.VBox([widgets.HTML(value=f"<b>Loaded trained step {int(model.checkpoint_identity['step'])}</b><br>Checkpoint SHA256: {html.escape(model.checkpoint_identity['sha256'])}"),
            self.status,self.game,widgets.HBox([self.start_button,self.step_button,self.play_button,self.stop_button]),
            widgets.HTML(value='<hr><b>Recorded replay — playback only</b>'),self.replay_path,
            widgets.HBox([self.load_button,self.replay_play,self.slider]),self.output])

    def _show(self,row,title):
        self.output.clear_output(wait=True)
        with self.output:
            render_ahr_decision(row,title)

    def _error(self,exc):
        self.status.value='<b>'+html.escape(f'{type(exc).__name__}: {exc}')+'</b>'

    def _start(self,_):
        try:
            if self.task is not None and not self.task.done():
                raise RuntimeError('Stop active autoplay and wait for the current decision before starting a new episode.')
            if self.player is not None and not self.player.finished:
                self.player.finish('user_started_new_episode')
            self.stop_requested=False
            self.player=FreshDemoPlayer(self.model,self.cfg,self.root,operation_mode='OFFLINE')
            observation=self.player.start(self.game.value)
            self.status.value='<b>FRESH public OFFLINE episode — actual environment, frozen trained checkpoint.</b>'
            initial={'observed_grid':observation['grid'],'next_observed_grid':observation['grid'],
                     'action':ActionSpec(0).to_dict(),'game_id':observation['game_id'],'state':observation['state'],
                     'levels_completed':observation['levels_completed'],'planning_trace':{},'mode':'fresh initial observation; no AHR action yet'}
            self._show(initial,'Fresh initial public observation')
        except Exception as exc:
            self._error(exc)

    def _step(self,_):
        try:
            if self.task is not None and not self.task.done():
                raise RuntimeError('Autoplay is active; use Stop before a manual step.')
            if self.player is None:
                raise RuntimeError('Press Start fresh first.')
            row=self.player.step()
            if row is not None:
                self._show(row,'FRESH: actual before/after observations and current AHR reasoning')
            if self.player.finished:
                self.status.value='<b>'+html.escape(f"Fresh episode ended: {self.player.manifest['status']}; actual state {self.player.manifest['actual_state']}")+'</b>'
        except Exception as exc:
            self._error(exc)

    async def _autoplay(self):
        try:
            while self.player is not None and not self.player.finished and not self.stop_requested:
                row=self.player.step()
                if row is not None:
                    self._show(row,'FRESH autoplay: real observed action outcome')
                await asyncio.sleep(.25)
            if self.player is not None and not self.player.finished:
                self.player.finish('user_stopped')
            if self.player is not None:
                self.status.value='<b>'+html.escape(f"Fresh episode ended: {self.player.manifest['status']}; actual state {self.player.manifest['actual_state']}")+'</b>'
        except Exception as exc:
            self._error(exc)

    def _play(self,_):
        try:
            if self.player is None or self.player.finished:
                raise RuntimeError('Start a fresh episode before autoplay.')
            if self.task is not None and not self.task.done():
                raise RuntimeError('Autoplay is already active.')
            self.stop_requested=False
            self.task=asyncio.create_task(self._autoplay())
        except Exception as exc:
            self._error(exc)

    def _stop(self,_):
        self.stop_requested=True
        if self.player is not None and not self.player.finished and (self.task is None or self.task.done()):
            self.player.finish('user_stopped')
        self.status.value='<b>Stop requested; it takes effect after the current model decision returns.</b>'

    def _load(self,_):
        try:
            if self.player is not None and not self.player.finished:
                raise RuntimeError('Stop fresh play before switching to a recorded replay.')
            path=Path(self.replay_path.value).expanduser()
            path=path if path.is_absolute() else self.root/path
            self.rows=load_watch_replay(path)
            self.slider.max=len(self.rows)-1;self.replay_play.max=len(self.rows)-1
            self.slider.value=0;self.replay_play.value=0
            self.status.value='<b>RECORDED REPLAY — no current environment actions or inference. Historical checkpoint identity is shown only if stored.</b>'
            self._show(self.rows[0],'RECORDED public transition; historical reasoning trace')
        except Exception as exc:
            self._error(exc)

    def _replay_step(self,change):
        if self.rows:
            self._show(self.rows[int(change['new'])],f"RECORDED public transition {int(change['new'])+1}/{len(self.rows)}")

def make_watch_dashboard(model,cfg,root=ROOT):
    from IPython.display import display
    dashboard=AHRWatchDashboard(model,cfg,root)
    display(dashboard.widget)
    return dashboard

## Later Run All: load trained weights and display controls
This cell allocates the full trained architecture on the configured device; it can use substantial memory and must be profiled. It refuses missing/untrained weights. No games are played until you use the fresh controls. For offline replay, paste an existing `reports/traces/*.jsonl` or `reports/demo/<episode_id>/` path. Replay from random data displays observations but no fabricated Queen or energy traces.

In [ ]:
cfg=load_config()
checkpoint_path=resolve_demo_checkpoint(cfg,root=ROOT)
demo_device=cfg.get('demo',{}).get('device',cfg['train']['device'])
demo_model=build_model_from_checkpoint(checkpoint_path,cfg=None,device=demo_device)
if int(demo_model.checkpoint_identity['step'])<=0:
    raise RuntimeError('A training step >0 is required for this trained-checkpoint demo.')
print({'checkpoint':demo_model.checkpoint_identity,'device':str(next(demo_model.parameters()).device),
       'status':'trained-weights-loaded-not-a-performance-result','fresh_actions_taken':0})
watch_dashboard=make_watch_dashboard(demo_model,cfg,root=ROOT)

## Interpret the display
The Worker proposes goal-directed candidates, the Scout proposes exploratory candidates, and the Queen allocates their bounded computation. The energy plot is conditional model scoring; the real before/after board establishes what actually happened. Lorentz events organize the learned workspace; their geometry does not establish unknown game rules. A fresh successful episode remains a public development demonstration, with its checkpoint hash, actual actions, timing and scope preserved in the journal. Use notebook 13 for frozen, family-held-out comparisons and separately disclosed official scoring.